## 1. Setup

### Install required packages

In [ ]:
%pip install google-generativeai python-dotenv

### Import libraries

In [15]:
import os
from dotenv import load_dotenv
import google.generativeai as genai
from google.api_core.exceptions import ResourceExhausted, InvalidArgument

print("Everything is working!")

Everything is working!


### Load your API key

In [16]:
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

print("API key loaded:", api_key is not None)

API key loaded: True


### Configure Gemini

In [17]:
genai.configure(api_key=api_key)

MODEL_NAME = "gemini-3-flash-preview"

print("Gemini configured successfully.")

Gemini configured successfully.


## 2. Create the Helper Function

messages → The conversation

model → Which Gemini model to use

temperature → controls how creative/random the answer is

In [18]:
def get_completion(messages, model=MODEL_NAME, temperature=0.7):
    """
    Sends messages to Gemini and returns the generated response.
    """

    # Extract system messages
    system_msgs = [
        m["content"]
        for m in messages
        if m["role"] == "system"
    ]

    system_instruction = "\n".join(system_msgs) if system_msgs else None

    # Remove system messages from chat history
    chat_msgs = [
        m for m in messages
        if m["role"] != "system"
    ]

    # Create Gemini model
    gen_model = genai.GenerativeModel(
        model_name=model,
        system_instruction=system_instruction
    )

    # Convert OpenAI-style roles to Gemini roles
    history = []

    for m in chat_msgs[:-1]:
        role = "model" if m["role"] == "assistant" else "user"

        history.append({
            "role": role,
            "parts": [m["content"]]
        })

    # Last message
    last_user_msg = chat_msgs[-1]["content"]

    try:
        chat = gen_model.start_chat(history=history)

        response = chat.send_message(
            last_user_msg,
            generation_config=genai.types.GenerationConfig(
                temperature=temperature
            )
        )

        return response.text

    except ResourceExhausted:
        return "⚠️ Gemini free-tier rate limit hit. Wait a minute and try again."

    except InvalidArgument as e:
        return f"⚠️ Invalid request: {e}"

    except Exception as e:
        return f"⚠️ Unexpected error: {e}"

### Test the function

In [19]:
messages = [
    {
        "role": "user",
        "content": "Say hello and tell me that you are working."
    }
]

response = get_completion(messages)

print(response)

Hello! I am working and ready to assist you.


## 3. LLM Architecture Messaging

We will demonstrate:

- System message
- User message
- Assistant response

In [20]:
messages = [
    {
        "role": "system",
        "content": "You are an expert italian Chef."
    },
    {
        "role": "user",
        "content": "How do I make a delicious pizza?"
    }
]

response = get_completion(messages)

print(response)

*Benvenuti!* Making a truly delicious pizza—a real *pizza napoletana* style—is not just about a recipe; it is about patience, high-quality ingredients, and respect for the dough.

As an Italian chef, I will guide you through the "Holy Trinity" of pizza: **The Dough (L'Impasto)**, **The Sauce (La Salsa)**, and **The Bake (La Cottura)**.

---

### 1. The Foundation: The Dough (L'Impasto)
Forget the "quick rise" recipes. For flavor and digestibility, you need time.

**Ingredients:**
*   **Flour:** 500g "00" Flour (finely milled) or a high-protein bread flour.
*   **Water:** 325ml (room temperature). This is 65% hydration.
*   **Salt:** 15g fine sea salt.
*   **Yeast:** 2g active dry yeast (or 1g if you have 24 hours).

**The Process:**
1.  **Mix:** Dissolve the yeast in the water. Gradually add the flour, mixing by hand until a shaggy dough forms. Add the salt last.
2.  **Knead:** Knead for about 10–15 minutes until the dough is smooth, elastic, and passes the "windowpane test" (you can s

## 4. Zero-shot Learning

Zero-shot learning means giving the model a task without providing examples.

In [21]:
zero_shot_prompt = """
Is this sentence Happy or Sad?

"I got a new puppy today!"
"""

messages = [
    {
        "role": "user",
        "content": zero_shot_prompt
    }
]

response = get_completion(messages, temperature=0)

print(response)

That sentence is **Happy**.


## 5. Few-shot Learning

Zero-shot learning means asking the model to perform a task without giving it any examples.

In [22]:
few_shot_prompt = """
Classify each fruit as Healthy or Unhealthy.

Fruit: Apple
Category: Healthy

Fruit: Candy
Category: Unhealthy

Fruit: Banana
Category:
"""

messages = [
    {
        "role": "user",
        "content": few_shot_prompt
    }
]

response = get_completion(messages, temperature=0)

print(response)

Fruit: Banana
Category: Healthy


## 6. Chain of Thought (CoT)

Chain of Thought prompting encourages the model to solve a problem step by step.

In [23]:
cot_prompt = """
I have 10 chocolates.
I give 3 chocolates to my friend.
Then I buy 2 more.

How many chocolates do I have?

Explain the calculation briefly.
"""

messages = [
    {
        "role": "user",
        "content": cot_prompt
    }
]

response = get_completion(messages)

print(response)

You have **9 chocolates**.

Here is the calculation:
1.  **Start:** 10 chocolates.
2.  **Give 3 away:** 10 - 3 = 7 chocolates.
3.  **Buy 2 more:** 7 + 2 = **9 chocolates**.


## 7. Tree of Thoughts (ToT)

Tree of Thoughts explores multiple possible solutions, compares them, and chooses the best option.

In [24]:
tot_prompt = """
I need to travel to university.

Consider these 3 options:

1. Bus
2. Train
3. Walking

For each option, give one advantage and one disadvantage.

Then choose the best option for a student
who wants to save money and time.
"""

messages = [
    {
        "role": "user",
        "content": tot_prompt
    }
]

response = get_completion(messages)

print(response)

Here are the advantages and disadvantages for each travel option:

### 1. Bus
*   **Advantage:** Most transit systems offer significant discounts for students, making it a very affordable motorized option.
*   **Disadvantage:** Buses are subject to road traffic and can be unreliable or slow during peak university hours.

### 2. Train
*   **Advantage:** It is generally the fastest way to travel long distances and avoids the unpredictability of traffic.
*   **Disadvantage:** It is typically the most expensive form of public transport and operates on a rigid schedule.

### 3. Walking
*   **Advantage:** It is completely free and serves as a reliable way to get daily exercise.
*   **Disadvantage:** It is the most time-consuming option and leaves you exposed to bad weather.

***

### The Best Option: The Bus
For a student who needs to balance **saving money and time**, the **Bus** is the best choice.

While walking saves the most money, it costs too much in time. While the train saves the mo